<a href="https://colab.research.google.com/github/floribert1/D-tection-Tumeur-/blob/main/tumeur.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from os import path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Dataset"

Mounted at /content/drive


In [7]:
train_datagen = ImageDataGenerator(
    rescale=1./255,          # normalisation 0–1
    rotation_range=20,       # rotation légère
    zoom_range=0.2,          # zoom
    horizontal_flip=True     # flip gauche/droite
)

In [8]:
train_generator = train_datagen.flow_from_directory(
    path,
    target_size=(224, 224),   # resize
    batch_size=32,
    class_mode='binary'       # tumeur / pas tumeur
)

Found 10093 images belonging to 2 classes.


In [9]:
val_datagen = ImageDataGenerator(rescale=1./255)

val_generator = val_datagen.flow_from_directory(
    path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

Found 10093 images belonging to 2 classes.


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [11]:
model = Sequential()

# Bloc 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# Bloc 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Bloc 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten (transforme 2D → 1D)
model.add(Flatten())

# Fully connected
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Sortie binaire
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=4,
    validation_data=val_generator,
    validation_steps=len(val_generator)
)

Epoch 1/4
240/316 ━━━━━━━━━━━━━━━━━━━━ 29:27 23s/step - accuracy: 0.8036 - loss: 0.5432

In [3]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy du modèle')
plt.legend(['train', 'validation'])
plt.show()